In [1]:
import os

os.environ["OPENAI_API_KEY"] = ""
os.environ["OPENAI_API_BASE"] = "https://openai.vocareum.com/v1"

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    max_retries=2,
    max_tokens=6000,
)

## Step 2: Generate Real Estate Listings

In [3]:
from pydantic import BaseModel, Field, NonNegativeInt, NonNegativeFloat
from langchain.output_parsers import PydanticOutputParser

# Pydantic model for structured data class
class RealEstateListing(BaseModel):
    neighborhood: str = Field(description="The neighborhood the listing is located")
    price: NonNegativeInt = Field(description="The price of the listing in USD")
    bedrooms: NonNegativeInt = Field(description="The number of bedrooms")
    bathrooms: NonNegativeFloat = Field(description="The number of bathrooms. Half bathrooms are supported")
    house_size: NonNegativeInt = Field(description="The square foot size of the dwelling")
    description: str

class RealEstateListingList(BaseModel):
    listings: list[RealEstateListing]

parser = PydanticOutputParser(pydantic_object=RealEstateListingList)

In [4]:
from langchain import PromptTemplate, LLMChain

template = """
Generate {num_listings} additional listings following these format instructions
{format_instructions}

These are 2 examples of raw data that is not yet formatted:
1)
Neighborhood: Green Oaks
Price: 800000
Bedrooms: 3
Bathrooms: 2
House Size: 2000
Description: Welcome to this eco-friendly oasis nestled in the heart of Green Oaks. This charming 3-bedroom, 2-bathroom home boasts energy-efficient features such as solar panels and a well-insulated structure. Natural light floods the living spaces, highlighting the beautiful hardwood floors and eco-conscious finishes. The open-concept kitchen and dining area lead to a spacious backyard with a vegetable garden, perfect for the eco-conscious family. Embrace sustainable living without compromising on style in this Green Oaks gem.

2)
Neighborhood: Sunny Ridge
Price: 600000
Bedrooms: 2
Bathrooms: 1.5
House Size: 1200
Description: A cozy retreat in the heart of Sunny Ridge! This charming 2-bedroom home offers modern updates throughout, featuring a spacious kitchen with quartz countertops, sleek appliances, and a sun-soaked living room. Enjoy quiet mornings on your private deck or explore the nearby hiking trails. Perfect for those seeking comfort and tranquility in a vibrant neighborhood.

You're job is to generate content like this and output them following the format instructions. It is crucial to follow the format or you will break the later processing and prevent people from buying new homes.
"""

prompt_template = PromptTemplate(
    template=template,
    input_variables=["num_listings"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

def generate_listings(num_listings):
    chain = LLMChain(llm=llm, prompt=prompt_template)
    response = chain.run(num_listings=num_listings)
    return parser.parse(response)


generated_listings = generate_listings(10)

/var/folders/1w/pwvs5lks2fjg2q9rq83vw1r00000gn/T/ipykernel_75404/831586055.py:34: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=prompt_template)
/var/folders/1w/pwvs5lks2fjg2q9rq83vw1r00000gn/T/ipykernel_75404/831586055.py:35: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = chain.run(num_listings=num_listings)


## Step 3: Store Listings in Vector DB

In [5]:
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.Client()
collection_name = "real_estate_listings"

if collection_name not in client.list_collections():
    collection = client.create_collection(name=collection_name)
else:
    collection = client.get_collection(name=collection_name)

In [6]:
from langchain_openai import OpenAIEmbeddings
embedding_model = OpenAIEmbeddings(model="text-embedding-3-large")

In [7]:
def ingest_listings(listings):
    documents = [
        f"Neighborhood: {l.neighborhood}\nPrice: {l.price}\nBedrooms: {l.bedrooms}\n"
        f"Bathrooms: {l.bathrooms}\nHouse Size: {l.house_size}\nDescription: {l.description}"
        for l in listings.listings
    ]
    metadatas = [listing.model_dump() for listing in listings.listings]
    ids = [f"{i}" for i in range(len(listings.listings))]
    embeddings = embedding_model.embed_documents(documents)
    collection.add(documents=documents, metadatas=metadatas, ids=ids, embeddings=embeddings)

ingest_listings(generated_listings)

## Step 4: Accept User Preferences

In [8]:
import gradio as gr

def create_interface(fn):
    interface = gr.Interface(
        fn=fn,
        inputs=[
            gr.Textbox(label="How big do you want your house to be (sqft)?"),
            gr.Textbox(label="What are 3 most important things for you in choosing this property?"),
            gr.Textbox(label="Which amenities would you like?"),
            gr.Textbox(label="Which transportation options are important to you?"),
            gr.Textbox(label="How urban do you want your neighborhood to be?"),
        ],
        outputs=gr.Textbox(lines=20, label="Matching Listings"),
        title="HomeMatch AI",
    )
    
    interface.launch()

## Step 5: Semantic Search Based on Preferences

In [9]:
def format_user_preferences(size, priorities, amenities, transport, urbanity):
    return f"I want my house to be around {size} squarefeet\n" \
                        f"My top priorities include: {priorities}\n" \
                        f"I prefer these amenities: {amenities}\n" \
                        f"I will rely on this transportation: {transport}\n" \
                        f"it should feel this level of urbanity: {urbanity}"
    

def get_user_preference_embeddings(user_preferences):
    return embedding_model.embed_query(user_preferences)

def search_similar_listings(user_embeddings):
    return collection.query(
        query_embeddings=[user_embeddings],
        n_results=5,
        include=["documents", "metadatas"]
    )

def find_matching_listings(size, priorities, amenities, transport, urbanity):
    user_preferences = format_user_preferences(size, priorities, amenities, transport, urbanity)
    user_embeddings = get_user_preference_embeddings(user_preferences)
    results = search_similar_listings(user_embeddings)

    formatted = []
    for metadata in results["metadatas"][0]:
        formatted.append(
            f"Neighborhood: {metadata['neighborhood']}\n"
            f"Price: ${metadata['price']}\n"
            f"Bedrooms: {metadata['bedrooms']}\n"
            f"Bathrooms: {metadata['bathrooms']}\n"
            f"Size: {metadata['house_size']} sqft\n"
            f"Description: {metadata['description']}\n"
        )

    return f"{'-'*40}\n".join(formatted)

create_interface(find_matching_listings)

Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


## Step 6: Personalize the Returned Listing

In [10]:
def find_matching_listings_personalized(size, priorities, amenities, transport, urbanity):
    user_preferences = format_user_preferences(size, priorities, amenities, transport, urbanity)
    user_embeddings = get_user_preference_embeddings(user_preferences)
    results = search_similar_listings(user_embeddings)

    orig_descriptions = "\n\n".join([result["description"] for result in results["metadatas"][0]])

    template = """
Based on these user_preferences:
{user_preferences}

Try to personalize each listing description. DO NOT make up random facts, just try to make it match what the user preferences are.

{descriptions}

The format to return in must be a string parsable by splitting on '\t' in python
"""

    prompt_template = PromptTemplate(
        template=template,
        input_variables=["user_preferences", "descriptions"],
    )

    chain = LLMChain(llm=llm, prompt=prompt_template)
    response = chain.run(user_preferences=user_preferences, descriptions=orig_descriptions)

    personalized_descriptions = response.split("\t")
    
    formatted = []
    for i in range(len(results["metadatas"][0])):
        metadata = results["metadatas"][0][i]
        personalized_description = personalized_descriptions[i]
        formatted.append(
            f"Neighborhood: {metadata['neighborhood']}\n"
            f"Price: ${metadata['price']}\n"
            f"Bedrooms: {metadata['bedrooms']}\n"
            f"Bathrooms: {metadata['bathrooms']}\n"
            f"Size: {metadata['house_size']} sqft\n"
            f"Description: {personalized_description}\n"
        )

    return f"{'-'*40}\n".join(formatted)

create_interface(find_matching_listings_personalized)

Running on local URL:  http://127.0.0.1:7861

To create a public link, set `share=True` in `launch()`.


## Step 8: Export Generated Listings

In [11]:
import json

with open("listings.json", "w") as file:
    json.dump(generated_listings.model_dump(), file, indent=2)